In [1]:
# Cell 1 — Imports + paths
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # notebook is inside /notebooks
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

BASELINE_DIR = ARTIFACTS_DIR / "baseline"
DISTILBERT_DIR = ARTIFACTS_DIR / "distilbert"   # rename if your folder differs

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Baseline exists:", BASELINE_DIR.exists())
print("DistilBERT exists:", DISTILBERT_DIR.exists())
print("Baseline files:", list(BASELINE_DIR.glob("*.*")))
print("DistilBERT files:", list(DISTILBERT_DIR.glob("*.*")))



PROJECT_ROOT: c:\Users\mansour\Documents\Clinical text classification
Baseline exists: True
DistilBERT exists: True
Baseline files: [WindowsPath('c:/Users/mansour/Documents/Clinical text classification/artifacts/baseline/confusion_matrix_dev.csv'), WindowsPath('c:/Users/mansour/Documents/Clinical text classification/artifacts/baseline/metrics_dev.json')]
DistilBERT files: [WindowsPath('c:/Users/mansour/Documents/Clinical text classification/artifacts/distilbert/confusion_matrix_dev.csv'), WindowsPath('c:/Users/mansour/Documents/Clinical text classification/artifacts/distilbert/metrics_dev.json')]


In [2]:
# Cell 2 — Load saved artifacts (metrics JSON + confusion CSV)
def load_run(run_dir: Path, split: str = "dev"):
    metrics_path = run_dir / f"metrics_{split}.json"
    cm_path = run_dir / f"confusion_matrix_{split}.csv"

    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing: {metrics_path}")
    if not cm_path.exists():
        raise FileNotFoundError(f"Missing: {cm_path}")

    with metrics_path.open("r", encoding="utf-8") as f:
        report = json.load(f)

    cm_df = pd.read_csv(cm_path, index_col=0)
    return report, cm_df

baseline_report, baseline_cm = load_run(BASELINE_DIR, split="dev")
distil_report, distil_cm = load_run(DISTILBERT_DIR, split="dev")

baseline_cm.shape, distil_cm.shape



((5, 5), (5, 5))

In [3]:
# Cell 3 — Helper functions: extract per-class metrics + summary
def report_to_df(report: dict) -> pd.DataFrame:
    """
    classification_report(output_dict=True) returns keys:
    class names + 'accuracy' + 'macro avg' + 'weighted avg'
    We'll keep only the class rows.
    """
    rows = []
    for k, v in report.items():
        if isinstance(v, dict) and all(m in v for m in ["precision", "recall", "f1-score"]):
            rows.append({"label": k, **v})
    df = pd.DataFrame(rows).set_index("label")
    return df

def overall_summary(report: dict) -> dict:
    return {
        "accuracy": report.get("accuracy", None),
        "macro_f1": report.get("macro avg", {}).get("f1-score", None),
        "weighted_f1": report.get("weighted avg", {}).get("f1-score", None),
    }

baseline_df = report_to_df(baseline_report)
distil_df = report_to_df(distil_report)

baseline_summary = overall_summary(baseline_report)
distil_summary = overall_summary(distil_report)

baseline_summary, distil_summary



({'accuracy': 0.8306633125910234,
  'macro_f1': 0.7666780857286796,
  'weighted_f1': 0.8280020346013901},
 {'accuracy': 0.8727326889977493,
  'macro_f1': 0.8116652580339142,
  'weighted_f1': 0.870180326686493})

In [5]:
# Cell 4 — Overall comparison table
overall = pd.DataFrame([
    {"model": "baseline_tfidf_logreg", **baseline_summary},
    {"model": "distilbert", **distil_summary},
]).set_index("model")

# Add deltas: distilbert - baseline
deltas = overall.loc["distilbert"] - overall.loc["baseline_tfidf_logreg"]
overall, deltas



(                       accuracy  macro_f1  weighted_f1
 model                                                 
 baseline_tfidf_logreg  0.830663  0.766678     0.828002
 distilbert             0.872733  0.811665     0.870180,
 accuracy       0.042069
 macro_f1       0.044987
 weighted_f1    0.042178
 dtype: float64)

In [6]:
# Cell 5 — Per-class comparison (precision/recall/F1) + deltas
# Align both dataframes on same label order
labels = sorted(set(baseline_df.index) & set(distil_df.index))

compare = pd.DataFrame(index=labels)
for metric in ["precision", "recall", "f1-score", "support"]:
    compare[f"baseline_{metric}"] = baseline_df.loc[labels, metric].astype(float)
    compare[f"distil_{metric}"] = distil_df.loc[labels, metric].astype(float)

# Deltas
compare["delta_precision"] = compare["distil_precision"] - compare["baseline_precision"]
compare["delta_recall"] = compare["distil_recall"] - compare["baseline_recall"]
compare["delta_f1"] = compare["distil_f1-score"] - compare["baseline_f1-score"]

compare.sort_values("delta_f1", ascending=False)


,baseline_precision,distil_precision,baseline_recall,distil_recall,baseline_f1-score,distil_f1-score,baseline_support,distil_support,delta_precision,delta_recall,delta_f1
BACKGROUND,0.650241,0.675521,0.665700,0.788924,0.657880,0.727832,3449.0,3449.0,0.025281,0.123224,0.069952
CONCLUSIONS,0.770018,0.845840,0.755565,0.814273,0.762723,0.829756,4582.0,4582.0,0.075822,0.058708,0.067033
macro avg,0.781918,0.836282,0.756863,0.802041,0.766678,0.811665,30212.0,30212.0,0.054364,0.045179,0.044987
weighted avg,0.827844,0.873999,0.830663,0.872733,0.828002,0.870180,30212.0,30212.0,0.046155,0.042069,0.042178
METHODS,0.878227,0.929068,0.928643,0.953031,0.902732,0.940897,9964.0,9964.0,0.050841,0.024388,0.038165
RESULTS,0.892336,0.914898,0.894421,0.932934,0.893377,0.923828,9841.0,9841.0,0.022562,0.038512,0.030450
OBJECTIVE,0.718768,0.816084,0.539983,0.521044,0.616679,0.636013,2376.0,2376.0,0.097317,-0.018939,0.019335


In [7]:
# Cell 6 — Focus on the key confusion pair: OBJECTIVE vs BACKGROUND
# (You can change labels if your dataset uses different names)
FOCUS_A = "OBJECTIVE"
FOCUS_B = "BACKGROUND"

focus = compare.loc[[FOCUS_A, FOCUS_B], [
    "baseline_precision","baseline_recall","baseline_f1-score",
    "distil_precision","distil_recall","distil_f1-score",
    "delta_precision","delta_recall","delta_f1"
]]
focus


,baseline_precision,baseline_recall,baseline_f1-score,distil_precision,distil_recall,distil_f1-score,delta_precision,delta_recall,delta_f1
OBJECTIVE,0.718768,0.539983,0.616679,0.816084,0.521044,0.636013,0.097317,-0.018939,0.019335
BACKGROUND,0.650241,0.665700,0.657880,0.675521,0.788924,0.727832,0.025281,0.123224,0.069952


In [8]:
# Cell 7 — Confusion matrix deltas (where did errors decrease/increase?)
# cm rows=true label, cols=pred label
cm_delta = distil_cm - baseline_cm

# Remove diagonal to focus on mistakes
cm_delta_no_diag = cm_delta.copy()
for lbl in cm_delta_no_diag.index:
    if lbl in cm_delta_no_diag.columns:
        cm_delta_no_diag.loc[lbl, lbl] = 0

# Biggest error reductions (most negative deltas) and increases (most positive)
stacked = cm_delta_no_diag.stack()

most_reduced = stacked.sort_values().head(10).reset_index()
most_reduced.columns = ["true_label", "pred_label", "delta_count"]  # negative = fewer mistakes

most_increased = stacked.sort_values(ascending=False).head(10).reset_index()
most_increased.columns = ["true_label", "pred_label", "delta_count"]  # positive = more mistakes

most_reduced, most_increased


(    true_label   pred_label  delta_count
 0      RESULTS      METHODS         -303
 1   BACKGROUND  CONCLUSIONS         -170
 2      METHODS      RESULTS         -135
 3   BACKGROUND    OBJECTIVE         -133
 4  CONCLUSIONS   BACKGROUND         -107
 5   BACKGROUND      METHODS         -106
 6    OBJECTIVE  CONCLUSIONS          -90
 7  CONCLUSIONS      METHODS          -79
 8    OBJECTIVE      METHODS          -70
 9      RESULTS  CONCLUSIONS          -69,
     true_label   pred_label  delta_count
 0    OBJECTIVE   BACKGROUND          222
 1   BACKGROUND   BACKGROUND            0
 2  CONCLUSIONS  CONCLUSIONS            0
 3    OBJECTIVE    OBJECTIVE            0
 4      METHODS      METHODS            0
 5      RESULTS      RESULTS            0
 6      RESULTS    OBJECTIVE           -1
 7      RESULTS   BACKGROUND           -6
 8   BACKGROUND      RESULTS          -16
 9    OBJECTIVE      RESULTS          -17)

In [9]:
# Cell 8 — Save final comparison tables to /artifacts/comparison
OUT_DIR = ARTIFACTS_DIR / "comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

overall.to_csv(OUT_DIR / "overall_metrics_dev.csv")
compare.to_csv(OUT_DIR / "per_class_metrics_dev.csv")
cm_delta.to_csv(OUT_DIR / "confusion_delta_dev.csv")

with (OUT_DIR / "notes_summary.txt").open("w", encoding="utf-8") as f:
    f.write("Baseline vs DistilBERT comparison (Dev)\n\n")
    f.write("Overall:\n")
    f.write(overall.to_string())
    f.write("\n\nFocus (OBJECTIVE/BACKGROUND):\n")
    f.write(focus.to_string())
    f.write("\n\nTop reduced confusions:\n")
    f.write(most_reduced.to_string(index=False))
    f.write("\n\nTop increased confusions:\n")
    f.write(most_increased.to_string(index=False))

print("Saved comparison outputs to:", OUT_DIR)


Saved comparison outputs to: c:\Users\mansour\Documents\Clinical text classification\artifacts\comparison
